# Basic Agentic Workflow

Helloooo everyone and welcome to an exciting lesson on Agentic AI! 🎉

Today, we're diving into **prompt chaining** agentic workflow pattern. What's that? It's just passing the output from one LLM to the next, step by step. Think of it like a relay race, but with prompts instead of batons.

We'll keep things super simple: manually run each cell, watch the magic happen, and see how chaining LLM calls lets us build more complex workflows.

Ready to see how agents can work together? Let's get started!

## As always, libraries first!

In [21]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

You can set your API Keys for each of the LLM providers using the following links:

- [OpenAI](https://platform.openai.com/api-keys)
- [Anthropic](https://console.anthropic.com/settings/keys)
- [Gemini](https://aistudio.google.com/app/apikey)

Once you have created the API Keys, you can store them on your `.env` file at the root of this repo

<div style="border-radius:16px;background:#2e3440;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
    <b style="color:#88c0d0;font-size:1.25em">Info:</b>
    <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
        <li>You can complete this entire notebook using just OpenAI models if you prefer!</li>
        <li>It's absolutely fine to skip Anthropic and Gemini for now — the workflow works perfectly with only OpenAI.</li>
        <li>Feel free to experiment with other providers later, but don't let missing API keys slow you down.</li>
    </ul>
    <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#88c0d0;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💡</div>
</div>

## The Workflow

```mermaid
graph LR
    A[Generate Tickets] --> B[Classify Priority] --> C[Respond to Tickets]
```

## Lets start with using OpenAI

In [12]:
# client
openai_client = OpenAI()

In [13]:
# messages list
message = "I want you to generate a customer support ticket for a 3rd party re-seller. "
message += "Each ticket should be a single sentence describing a common issue a customer might have with a product or service. "
message += "Please ensure the ticket is varied and covers different types of problems. "
message += "Do not give any subjects, only the body of the ticket."

messages = [{"role": "user", "content": message}]

In [14]:
# response

openai_response = openai_client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

ticket = openai_response.choices[0].message.content
#print(f"### Generated Ticket:\n{ticket}")
display(Markdown(f"### Generated Ticket:\n{ticket}"))

### Generated Ticket:
The product I received is missing several accessories that were advertised in the listing.

I love markdown. It is a lightweight method of rendering and formatting text that is super versatile without having to use heavy softwares like MS Word or Google Docs.

You can learn more about markdown sytanx [here](https://www.markdownguide.org/basic-syntax/)

A really informative YouTube video talking about the [Unreasonable Effectiveness of Plain Text](https://www.youtube.com/watch?v=WgV6M1LyfNY)

## Lets pass these on to an Anthropic model and ask it to classify the priority level of each ticket

In [15]:
# anthropic client, pass in base_url
anthropic_client = OpenAI(api_key=ANTHROPIC_API_KEY, base_url="https://api.anthropic.com/v1")

In [16]:
# messages list
message = "I want you to classify the priority of the following customer support ticket. "
message += "The ticket is as follows: "+ ticket + " "
message += "Please classify the priority as either 'Low', 'Medium', or 'High'. "
message += "Respond with only the priority level."

messages = [{"role": "user", "content": message}]

In [17]:
# response

anthropic_response = anthropic_client.chat.completions.create(  
    #model="claude-3-5-haiku-latest",
    model="claude-3-haiku-20240307",
    messages=messages
)

priority = anthropic_response.choices[0].message.content
display(Markdown(f"### Classified Priority:\n{priority}"))

### Classified Priority:
Medium

## Now Gemini should determine the appropriate response

In [22]:
# gemini client
gemini_client = OpenAI(api_key=GEMINI_API_KEY, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [23]:
# messages list

message = "You are to determine an appropriate response to the following customer support ticket. "
message += "The ticket is as follows: "+ ticket + " "
message += "The priority level of this ticket is: " + priority + " "
message += "Please provide a response that addresses the customer's issue in a short and concise manner. "
    
messages = [{"role": "user", "content": message}]

In [ ]:
# response

gemini_response = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages
)

response = gemini_response.choices[0].message.content
display(Markdown(f"### Generated Response:\n{response}"))

### Generated Response:
Subject: Regarding your recent order - Missing accessories

Dear [Customer Name],

We apologize for the missing accessories in your recent order. This is certainly not the experience we want for our customers.

Please reply with your order number and a list of the accessories that were missing, and we will promptly arrange for them to be sent to you.

Thank you for your patience and understanding.

Sincerely,

[Your Name/Company Name] Support Team

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
    <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
    <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6"></ul>
        <li>Recreate the customer support ticket workflow using an <b>evaluator-optimizer agentic workflow pattern</b> instead of prompt chaining.</li>
        <li>Your evaluator agent should assess the quality and completeness of each ticket and suggest improvements.</li>
        <li>Your optimizer agent should revise the tickets based on evaluator feedback, aiming for clarity and actionable details.</li>
        <li>Try to implement this using at least two LLM calls (one for evaluation, one for optimization) and display the before/after results.</li>
        <li>Share your work in the community-contributions folder by creating a folder with your name. Eg. shaheer-airaj.</li>
    </ul>
    <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

In [38]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

# client
openai_client = OpenAI()

# messages list
message = "I want you to generate 3 customer support tickets for a 3rd party re-seller. "
message += "Each ticket should be a single sentence describing a common issue a customer might have with a product or service. "
message += "Please ensure the tickets are varied and cover different types of problems. "
message += "Return each ticket on a new line. "
message += "Do not give any subjects, only the body of the ticket."

messages = [{"role": "user", "content": message}]

openai_response = openai_client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

# -------- Generate Ticket  --------
tickets_text = openai_response.choices[0].message.content.strip()

tickets = [t.strip() for t in tickets_text.split("\n") if t.strip()]

display(Markdown("## Generated Tickets"))

for i, ticket in enumerate(tickets, 1):
    display(Markdown(f"---\n## Ticket {i} (Before)\n{ticket}"))

    # -------- Evaluator Agent --------
    evaluation_prompt = f"""
You are an evaluator agent for customer support tickets.

Evaluate the quality of the following ticket.

Focus on:
- clarity
- completeness
- actionable information
- missing context

Return:
1. Short evaluation
2. Missing information
3. Suggestions for improvement

Ticket:
{ticket}
"""

    evaluation_messages = [
        {"role": "system", "content": "You are a strict evaluator agent."},
        {"role": "user", "content": evaluation_prompt}
    ]

    evaluation_response = openai_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=evaluation_messages
    )

    evaluation = evaluation_response.choices[0].message.content.strip()

    display(Markdown(f"### Evaluator Feedback\n{evaluation}"))

    # -------- Optimizer Agent --------
    optimizer_prompt = f"""
You are an optimizer agent.

Rewrite the ticket using the evaluator feedback.

Goals:
- clearer problem description
- more actionable details
- still a single sentence

Original ticket:
{ticket}

Evaluator feedback:
{evaluation}

Return ONLY the improved ticket.
"""

    optimizer_messages = [
        {"role": "system", "content": "You improve support tickets."},
        {"role": "user", "content": optimizer_prompt}
    ]

    optimizer_response = openai_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=optimizer_messages
    )

    optimized_ticket = optimizer_response.choices[0].message.content.strip()

    display(Markdown(f"## Ticket {i} (After)\n{optimized_ticket}"))

## Generated Tickets

---
## Ticket 1 (Before)
The product I received was damaged and Missing parts.

### Evaluator Feedback
1. Short evaluation:
The ticket lacks clarity and detail, making it difficult to understand the specific issue, what parts are missing, and the extent of the damage. It also lacks actionable information for resolution.

2. Missing information:
- Specific product details (model, order number)
- Description of the damage
- Which parts are missing
- Photos of the damaged/missing parts
- Preferred resolution (refund, replacement, repair)

3. Suggestions for improvement:
- Provide detailed product information, including order number and model
- Clearly describe the damage and specify missing parts
- Include photographs to illustrate the issue
- State your preferred resolution to enable appropriate action

## Ticket 1 (After)
My order #12345 for a Model X blender arrived with significant damage to the lid and missing the user manual; I have attached photos and prefer a replacement.

---
## Ticket 2 (Before)
I'm unable to track my order despite receiving the confirmation email.

### Evaluator Feedback
1. Short evaluation:
The ticket lacks clarity and detail. It states the issue broadly but does not specify which order, what steps have been taken, or what tracking information is missing.

2. Missing information:
- Order number or confirmation email details
- Specific tracking information expected
- Confirmation of whether the order was successfully shipped
- Any error messages or issues encountered when attempting to track

3. Suggestions for improvement:
- Include the order number or relevant confirmation details
- Mention the date of the order or confirmation email
- Specify the platform or link used to track the order
- Describe any error messages or unusual behavior observed
- Clarify what tracking information is missing or what outcome is expected

## Ticket 2 (After)
I cannot track my order #123456 placed on October 10, 2023, using the link provided, as it shows no shipping updates or error messages, despite receiving a confirmation email.

---
## Ticket 3 (Before)
My digital download is not working even after multiple attempts.

### Evaluator Feedback
1. Short evaluation:
The ticket lacks clarity and detailed information, making it difficult to diagnose the issue. It is incomplete and does not specify the product, steps already taken, or error messages. No actionable information is provided for support to assist effectively.

2. Missing information:
- Specific product or download
- Error messages or symptoms
- Details of attempted solutions
- Device or platform details
- Purchase or account information

3. Suggestions for improvement:
- Clearly specify the product or digital item affected.
- Describe any error messages received or specific issues encountered.
- List the troubleshooting steps already attempted.
- Provide details about the device, operating system, or platform used.
- Include relevant account or purchase details for verification.

## Ticket 3 (After)
My digital download for the "E-Book Series" on Windows 10 is not opening, showing error code 503 after multiple download attempts from my account (username: user123), and I have already cleared browser cache and restarted my device.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

# client
openai_client = OpenAI()

# messages list
message = "Szeretném, ha generálnál 3 ügyfélszolgálati jegyet (support ticketet) egy harmadik fél viszonteladó számára. "
message += "Minden ticket egyetlen mondat legyen, amely leír egy gyakori problémát, amellyel egy ügyfél találkozhat egy termékkel vagy szolgáltatással kapcsolatban. "
message += "Kérlek biztosítsd, hogy a ticketek változatosak legyenek és különböző problématípusokat fedjenek le. "
message += "Minden ticket új sorba kerüljön. "
message += "Ne adj meg tárgyat (subject), csak a ticket szövegét."

messages = [{"role": "user", "content": message}]

openai_response = openai_client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

# -------- Generate Ticket  --------
tickets_text = openai_response.choices[0].message.content.strip()

tickets = [t.strip() for t in tickets_text.split("\n") if t.strip()]

display(Markdown("## Generált Ticketek"))

for i, ticket in enumerate(tickets, 1):
    display(Markdown(f"---\n## Ticket {i} (Előtte)\n{ticket}"))

    # -------- Evaluator Agent --------
    evaluation_prompt = f"""
Te egy értékelő agent vagy ügyfélszolgálati ticketekhez.

Értékeld a következő ticket minőségét.

Fókuszálj a következőkre:
- érthetőség
- teljesség
- cselekvésre alkalmas információ
- hiányzó kontextus

Add vissza:
1. Rövid értékelés
2. Hiányzó információk
3. Fejlesztési javaslatok

Ticket:
{ticket}
"""

    evaluation_messages = [
        {"role": "system", "content": "Szigorú értékelő agent vagy."},
        {"role": "user", "content": evaluation_prompt}
    ]

    evaluation_response = openai_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=evaluation_messages
    )

    evaluation = evaluation_response.choices[0].message.content.strip()

    display(Markdown(f"### Értékelő visszajelzés\n{evaluation}"))

    # -------- Optimizer Agent --------
    optimizer_prompt = f"""
Te egy optimalizáló agent vagy.

Írd át a ticketet az értékelő visszajelzése alapján.

Célok:
- világosabb probléma leírás
- több cselekvésre alkalmas részlet
- továbbra is egyetlen mondat legyen

Eredeti ticket:
{ticket}

Értékelő visszajelzés:
{evaluation}

CSA K a javított ticketet add vissza.
"""

    optimizer_messages = [
        {"role": "system", "content": "Ügyfélszolgálati ticketeket javítasz."},
        {"role": "user", "content": optimizer_prompt}
    ]

    optimizer_response = openai_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=optimizer_messages
    )

    optimized_ticket = optimizer_response.choices[0].message.content.strip()

    display(Markdown(f"## Ticket {i} (Utána)\n{optimized_ticket}"))